# RFModelQ3 — Hyperparameter Selection (Random Forest)

## Setup

In [1]:
import joblib
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline

PART_B_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "data" / "yelp_review_full_raw_30k.csv").exists())
DATA_FILE = PART_B_DIR / "data" / "yelp_clean.csv"
MODEL_DIR = PART_B_DIR / "models"

## Load and split

In [2]:
# --- Same load and split as Q2, so the tuned model is measured on the identical test set ---
df = pd.read_csv(DATA_FILE, usecols=["clean_text", "sentiment"])
df = df.dropna(subset=["clean_text", "sentiment"]).copy()
df["clean_text"] = df["clean_text"].astype(str).str.strip()
df = df[df["clean_text"] != ""]

X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["sentiment"],
    test_size=0.2, random_state=42, stratify=df["sentiment"]
)
print(f"Train: {len(X_train)}   Test: {len(X_test)}")

Train: 23998   Test: 6000


## Base pipeline

In [3]:
# --- Pipeline to tune. strip_accents is fixed; every other setting is searched below. ---
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(strip_accents="unicode")),
    ("clf", RandomForestClassifier(random_state=42, n_jobs=-1)),
])

## Search space

In [4]:
# --- Two groups of hyperparameters: how the text is represented, and how the forest is grown ---
param_dist = {
    "tfidf__max_features": [10000, 20000, None],   # vocabulary cap
    "tfidf__ngram_range": [(1, 1), (1, 2)],        # bigrams capture phrases like "not good"
    "tfidf__min_df": [1, 2, 3],                    # discard rare / typo terms
    "tfidf__sublinear_tf": [True, False],          # log-scale term frequency
    "clf__n_estimators": [100, 200, 300],          # number of trees averaged
    "clf__max_depth": [None, 30],                  # tree size limit
    "clf__min_samples_leaf": [1, 2],               # leaf smoothing
    "clf__class_weight": [None, "balanced", "balanced_subsample"],  # 2:1:2 class imbalance
}

## Randomized search

In [5]:
# --- 10 sampled configurations x 3-fold cross validation = 30 fits.
#     Scored on macro F1 so the smaller neutral class counts as much as the other two.
#     n_jobs=1 on the search itself because RandomForestClassifier already uses all cores.
search = RandomizedSearchCV(
    pipeline, param_dist,
    n_iter=10, cv=3,
    scoring="f1_macro",
    random_state=42, n_jobs=1, verbose=1,
)
search.fit(X_train, y_train)
print("Search complete.")

Fitting 3 folds for each of 10 candidates, totalling 30 fits


Search complete.


## Best configuration

In [6]:
# --- Report the winning configuration and its cross-validated score ---
print("=== Q3: Best Hyperparameters ===")
print("=" * 48)
for param, value in sorted(search.best_params_.items()):
    # Keep a "tfidf_" prefix on vectoriser params so they stay distinguishable
    # from the classifier params in the printout.
    name = param.replace("clf__", "").replace("tfidf__", "tfidf_")
    print(f"  {name:<24}: {value}")
print("=" * 48)
print(f"Best CV Macro F1: {search.best_score_:.4f}")

=== Q3: Best Hyperparameters ===
  class_weight            : balanced
  max_depth               : 30
  min_samples_leaf        : 2
  n_estimators            : 300
  tfidf_max_features      : None
  tfidf_min_df            : 3
  tfidf_ngram_range       : (1, 1)
  tfidf_sublinear_tf      : True
Best CV Macro F1: 0.6474


## Baseline vs tuned

In [7]:
# --- Did the search actually help? Score the untuned Q2 model and the tuned model
#     on the same held-out test set. ---
baseline = joblib.load(MODEL_DIR / "rf_pipeline.joblib")
tuned = search.best_estimator_

scores = {}
print(f"{'Model':<24}{'Accuracy':>10}{'Macro F1':>10}")
for name, model in [("Q2 baseline (defaults)", baseline), ("Q3 tuned", tuned)]:
    p = model.predict(X_test)
    scores[name] = (accuracy_score(y_test, p), f1_score(y_test, p, average="macro"))
    print(f"{name:<24}{scores[name][0]:>10.4f}{scores[name][1]:>10.4f}")

gain_acc = scores["Q3 tuned"][0] - scores["Q2 baseline (defaults)"][0]
gain_f1 = scores["Q3 tuned"][1] - scores["Q2 baseline (defaults)"][1]
print(f"{'Improvement':<24}{gain_acc:>+10.4f}{gain_f1:>+10.4f}")

Model                     Accuracy  Macro F1


Q2 baseline (defaults)      0.7050    0.5407


Q3 tuned                    0.7040    0.6561
Improvement                -0.0010   +0.1153


## Save tuned model

In [8]:
# --- Persist the tuned pipeline for the Q4 evaluation ---
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(tuned, MODEL_DIR / "rf_tuned.joblib")
print(f"Saved: {MODEL_DIR / 'rf_tuned.joblib'}")

Saved: C:\Users\yingx\Desktop\TextAssignment\Part_B\models\rf_tuned.joblib
